# Track-map outline spike — derive the Tempelhof circuit from GPS

**Goal:** derive a clean *GPS-derived SVG* track outline for the Race-Day Companion.
Take one car's 20 Hz GPS telemetry, project lat/lng to flat metres, extract one clean
lap, and emit an SVG `<path>` + the projection transform to `track_outline.json`.

**Why GPS, not the circuit PDF:** cars and track come from the *same* coordinate source,
so the live car dots sit on the track by construction — no manual georeferencing.

Validated against Berlin R10, car 13 — traces Tempelhof cleanly. Run top-to-bottom in
Colab Enterprise (needs read access to gs://class-demo).

In [ ]:
%pip install -q gcsfs pyarrow pandas numpy matplotlib

In [ ]:
import json, math, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import google.auth

_, PROJECT_ID = google.auth.default()
print("Project:", PROJECT_ID)

CAR = 13   # which car's lap to trace — any car that runs the full race works (13 = Da Costa)
TELEM_DIR = "gs://class-demo/formula-e/berlin_2024/r10/telemetry/"  # 20 Hz, partitioned by car_number
GREEN_FLAG = pd.Timestamp("2024-05-12 13:04:05.726")  # R10 green flag (UTC); repo-wide constant

## 1. Load one car's GPS telemetry

In [ ]:
try:
    df = pd.read_parquet(TELEM_DIR, filters=[("car_number", "==", CAR)], engine="pyarrow")
    if df.empty:  # partition value might be stored as a string
        df = pd.read_parquet(TELEM_DIR, filters=[("car_number", "==", str(CAR))], engine="pyarrow")
except Exception as e:
    print("partitioned read failed, listing blobs instead:", e)
    from google.cloud import storage
    cli = storage.Client(project=PROJECT_ID)
    bucket, prefix = "class-demo", "formula-e/berlin_2024/r10/telemetry/"
    blobs = [b.name for b in cli.list_blobs(bucket, prefix=prefix) if b.name.endswith(".parquet")]
    print(f"{len(blobs)} parquet blobs; sample:", blobs[:3])
    hit = [b for b in blobs if f"car_number={CAR}" in b or f"/{CAR}." in b or f"_{CAR}." in b]
    df = pd.concat([pd.read_parquet(f"gs://{bucket}/{b}") for b in (hit or blobs[:1])], ignore_index=True)

print("shape:", df.shape)
print("columns:", list(df.columns))
df.head(3)

## 2. Sort by time, drop the pre-race window, pull the GPS columns
The export starts ~12:58 UTC with the car stationary on the grid; we keep only from the
green flag onward so the lap extraction starts from a moving car.

In [ ]:
tcol = next((c for c in df.columns if re.search(r"ts|time|utc|wall|elapsed", c, re.I)), None)
print("time column:", tcol)
if tcol:
    df = df.sort_values(tcol)
    df = df[pd.to_datetime(df[tcol]) >= GREEN_FLAG]   # drop pre-race grid sitting

lat = df["tv_gps_lat"].to_numpy(float)
lng = df["tv_gps_long"].to_numpy(float)
ok = np.isfinite(lat) & np.isfinite(lng) & (lat != 0) & (lng != 0)
lat, lng = lat[ok], lng[ok]
print(f"{len(lat)} GPS points | lat {lat.min():.5f}..{lat.max():.5f} | lng {lng.min():.5f}..{lng.max():.5f}")

## 3. Project lat/lng → flat metres (local equirectangular)
Tempelhof is ~2 km and flat, so a local equirectangular projection about the mean latitude
is accurate to well under a metre. The live UI uses this SAME transform for car dots, so
we save its parameters.

In [ ]:
lat0, lng0 = float(lat.mean()), float(lng.mean())
M_PER_DEG_LAT = 110540.0
M_PER_DEG_LNG = 111320.0 * math.cos(math.radians(lat0))
x = (lng - lng0) * M_PER_DEG_LNG
y = (lat - lat0) * M_PER_DEG_LAT

plt.figure(figsize=(8, 8))
plt.plot(x, y, lw=0.3)
plt.gca().set_aspect("equal")
plt.title(f"All GPS points, car {CAR} (every lap overlaid = the circuit)")
plt.show()

## 4. Extract ONE clean lap
Anchor ~30% into the race (clean racing rhythm), walk forward until the trace returns near
the anchor AFTER covering a full lap of distance, then trim any GPS-dropout "teleport"
seams (a single huge jump between 20 Hz samples) by keeping the longest contiguous arc.

In [ ]:
seg = np.hypot(np.diff(x), np.diff(y))
cum = np.concatenate([[0.0], np.cumsum(seg)])          # metres travelled along the trace
start = int(np.searchsorted(cum, cum[-1] * 0.30))      # ~30% in
ax_, ay_ = x[start], y[start]
d = np.hypot(x - ax_, y - ay_)
LAP_MIN_M = 1500                                       # must travel >= this before "back to start"
ret = next((i for i in range(start + 1, len(x))
            if (cum[i] - cum[start]) > LAP_MIN_M and d[i] < 30.0), len(x) - 1)
lx, ly = x[start:ret + 1], y[start:ret + 1]

# despike: cut at any jump > 25 m between consecutive 20 Hz samples (a dropout, not racing),
# keep the longest contiguous arc
jump = np.hypot(np.diff(lx), np.diff(ly))
breaks = np.where(jump > 25.0)[0]
if len(breaks):
    bounds = [0, *(breaks + 1).tolist(), len(lx)]
    a, b = max(((bounds[i], bounds[i + 1]) for i in range(len(bounds) - 1)),
               key=lambda s: s[1] - s[0])
    lx, ly = lx[a:b], ly[a:b]
    print(f"trimmed {len(breaks)} dropout seam(s)")

print(f"lap slice: {len(lx)} points, {float(np.hypot(np.diff(lx), np.diff(ly)).sum()):.0f} m")
plt.figure(figsize=(8, 8))
plt.plot(lx, ly, lw=1.0)
plt.gca().set_aspect("equal")
plt.title(f"One lap, car {CAR}")
plt.show()

## 5. Build the SVG path + viewBox (and save the transform)
Downsample to a few hundred points, normalise into an SVG viewBox (y flipped — SVG y grows
downward), and save everything the live UI needs to place cars with the same transform.

In [ ]:
N = 400
idx = np.linspace(0, len(lx) - 1, min(N, len(lx))).astype(int)
sx, sy = lx[idx], ly[idx]

pad = 30.0
minx, maxx, miny, maxy = float(sx.min()), float(sx.max()), float(sy.min()), float(sy.max())
W, H = maxx - minx, maxy - miny

def to_svg(px, py):
    return (px - minx) + pad, (maxy - py) + pad   # flip y for screen coords

pts = [to_svg(px, py) for px, py in zip(sx, sy)]
path = "M " + " L ".join(f"{px:.1f} {py:.1f}" for px, py in pts) + " Z"
viewBox = f"0 0 {W + 2 * pad:.1f} {H + 2 * pad:.1f}"
print("viewBox:", viewBox)
print("path points:", len(pts))
print("path head:", path[:120], "...")

plt.figure(figsize=(8, 8))
plt.plot([p[0] for p in pts], [p[1] for p in pts])
plt.gca().invert_yaxis()
plt.gca().set_aspect("equal")
plt.title("SVG-space preview (how it will render in the UI)")
plt.show()

In [ ]:
out = {
    "race_id": "berlin_2024_r10",
    "viewBox": viewBox,
    "path": path,
    # The live UI projects each car's (lat,lng) with these, then maps to SVG via
    #   x = (lng-lng0)*m_per_deg_lng ; y = (lat-lat0)*m_per_deg_lat
    #   svg_x = (x-minx)+pad ; svg_y = (maxy-y)+pad
    "transform": {
        "lat0": lat0, "lng0": lng0,
        "m_per_deg_lat": M_PER_DEG_LAT, "m_per_deg_lng": M_PER_DEG_LNG,
        "minx": minx, "maxy": maxy, "pad": pad,
    },
}
with open("track_outline.json", "w") as f:
    json.dump(out, f, indent=2)
print("wrote track_outline.json")

## Done
`track_outline.json` holds the SVG `path` + `viewBox` + the projection `transform`. Drop it
into `frontend/static/` and the companion page renders the outline and plots live car dots
with the same transform (so cars sit on the track). Re-run with a different `CAR` if a lap is
ever dirty.